In [ ]:
import glob
import os

import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba

In [ ]:
logs_dir = "button_logs"
participant_files = glob.glob(os.path.join(logs_dir, "*"))

excluded_participants = ['02', '07', '14', '19', '28']  

participant_files = [
    path
    for path in participant_files
    if os.path.basename(path) not in excluded_participants
]

In [ ]:
rows = []

for participant_file in participant_files:
    participant_id = os.path.basename(participant_file)

    for csv_file in glob.glob(os.path.join(participant_file, "*.csv")):
        data = (
            pd.read_csv(csv_file)
            .drop(columns="timestamp")
            .drop_duplicates()
        )

        if data.empty or data["condition"].iat[0] != "Lumo":
            continue

        route = data["route"].iat[0]
        clicked = set(
            data.loc[
                (data["button"] == "Agent") & (data["zone"].isin(["A", "B", "C", "D"])),
                "zone",
            ]
        )

        rows.extend(
            {
                "participant_id": participant_id,
                "route": route,
                "zone": zone,
                "click": int(zone in clicked),
            }
            for zone in ["A", "B", "C", "D"]
        )

clicks = pd.DataFrame(rows)
clicks

In [ ]:
clicks_counts = clicks.groupby(["participant_id"])["click"].sum().reset_index(name="clicks")
clicks_counts

Participant counts per route are calculated from the filtered logs.

In [ ]:
clicks_per_event = clicks.groupby(["route", "zone"])["click"].sum().reset_index(name="clicks")
clicks_per_event

In [ ]:
total_number_of_participants = len(participant_files)
print(f"Total number of participants: {total_number_of_participants}")
print(f"Total number of clicks: {clicks_counts['clicks'].sum()} of possible {len(participant_files) * 4} (4 per participant)")
print(f"Percentage of clicks: {(clicks_counts['clicks'].sum() / (len(participant_files) * 4)) * 100:.2f}%")
print("Median number of clicks per participant: {:.2f}".format(clicks_counts['clicks'].median()))
print("Mean number of clicks per participant: {:.2f}".format(clicks_counts['clicks'].mean()))
print("Standard deviation of clicks per participant: {:.2f}".format(clicks_counts['clicks'].std()))

print(f"Sum of clicks per route and zone:\n{clicks_per_event.pivot(index='route', columns='zone', values='clicks')}")



In [ ]:
participants_per_route = (
    clicks.groupby("route", as_index=False)["participant_id"]
    .nunique()
    .rename(columns={"participant_id": "n_unique_participants"})
)
participants_per_route

Adding Explanations and Categories:

Events involving dynamic obstacles or evasive maneuvers: high safety relevance, whereas events involving static obstacles or
sensor uncertainty are classified as having low safety relevance.


 Cause perceptibility is considered hard to perceive when
the underlying reason for the vehicle’s behavior is centrally visible in the field of view and/or accompanied by salient
auditory cues. Conversely, it is considered easy to perceive when the cause occupies only a small portion of the visual
field, appears in the peripheral field of view, or is not directly observable.

In [ ]:

explanations = pd.DataFrame([
    ["A", "A", "The lane ahead is blocked.", "I'm recalculating my route to pass safely.", "Low", "Hard", "Static Obstacle"],
    ["A", "B", "There is an object in front of the car.", "I'm stopping to assess the situation.", "Low", "Hard", "Sensor Uncertainty"],
    ["A", "C", "Another vehicle pulled out suddenly.", "I performed an emergency brake.", "High", "Easy", "Evasive Maneuver"],
    ["A", "D", "I'm detecting a pedestrian close to the road.", "I'm stopping to assess the situation.", "High", "Hard", "Dynamic Obstacle"],

    ["B", "A", "A pedestrian is crossing the road.", "I'm stopping and will proceed once the road ahead is clear.", "High", "Easy", "Dynamic Obstacle"],
    ["B", "B", "The lane ahead is obstructed by a larger object.", "I'm recalculating my route to pass safely.", "Low", "Easy", "Static Obstacle"],
    ["B", "C", "I can't determine the traffic light's state.", "I'm stopping to assess the situation.", "Low", "Hard", "Sensor Uncertainty"],
    ["B", "D", "An emergency vehicle is approaching.", "I'm changing lanes to let it pass.", "High", "Easy", "Evasive Maneuver"],

    ["C", "A", "Another vehicle ran a red light.", "I performed an emergency brake.", "High", "Easy", "Evasive Maneuver"],
    ["C", "B", "There is a jogger on the road.", "I'm following at a safe distance until the road is clear.", "High", "Hard", "Dynamic Obstacle"],
    ["C", "C", "There's an object ahead.", "I'm stopping to assess the situation.", "Low", "Hard", "Static Obstacle"],
    ["C", "D", "The car ahead is not moving.", "I'm stopping to assess the situation.", "Low", "Easy", "Sensor Uncertainty"]
], columns=["route", "zone", "why", "how", "safety_relevance", "perceivability", "Obstacle Type"])


click_merged = (
    clicks_per_event
    .merge(explanations, on=["route", "zone"], how="left")
    .merge(participants_per_route, on="route", how="left")
)
click_merged["percentage_of_clicks"] = (
    click_merged["clicks"] / click_merged["n_unique_participants"]
) * 100

Barplot

In [ ]:
click_merged = click_merged.sort_values(["route", "zone"]).copy()
click_merged["label"] = click_merged["why"]

hatch_map = {
    "Hard": "///",
    "Easy": "",
}

hatches = (
    click_merged["perceivability"]
    .map(hatch_map)
    .fillna("")
    .tolist()
)

facecolors = [
    to_rgba("#6929c4", alpha=0.5),
    to_rgba("#005d5d", alpha=0.5),
]

mpl.rcParams.update({
    "font.family": "Avenir",
    "font.sans-serif": ["Avenir"],
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

fig, ax = plt.subplots(figsize=(20, 4), dpi=100)

ax.grid(False)                
x = range(len(click_merged))

ax.bar(
    x,
    click_merged["percentage_of_clicks"],
    color=facecolors,
    edgecolor="none",
)

outline_bars = ax.bar(
    x,
    click_merged["percentage_of_clicks"],
    facecolor="none",
    edgecolor="black",
    linewidth=0.8,
)

for bar, hatch in zip(outline_bars, hatches):
    bar.set_hatch(hatch)

ax.set_xticks(list(x))
ax.set_xticklabels(click_merged["label"])

handles = [
    mpatches.Patch(
        facecolor=facecolors[0],
        edgecolor="black",
        label="Lower safety relevance",
    ),
    mpatches.Patch(
        facecolor=facecolors[1],
        edgecolor="black",
        label="Higher safety relevance",
    ),
    mpatches.Patch(
        facecolor="white",
        edgecolor="black",
        hatch=hatch_map["Hard"],
        linewidth=0.8,
        label="Hard to perceive",
    ),
    mpatches.Patch(
        facecolor="white",
        edgecolor="black",
        hatch=hatch_map["Easy"],
        linewidth=0.8,
        label="Easy to perceive",
    ),
]

ax.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
)

ax.tick_params(axis="x", rotation=45)

for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")

ax.set_ylabel("Explanation requests [%]")

output_dir = "plots"

fig.savefig(
    os.path.join(output_dir, "click_pct.pdf"),
    format="pdf",
    bbox_inches="tight",
)

plt.close(fig)

Save clicks for GLMM models (see exploratory_pull_analysis.R)

In [ ]:
clicks.merge(explanations, on=["route", "zone"], how="left").to_csv("../DATA/clicks.csv", index=False)
